# CV Masterclass 10: Production Augmentations

Welcome to Notebook 10. The pristine academic datasets (ImageNet, COCO) are a lie. They are perfectly lit, high-resolution `.png` files captured by professional cameras.

Production data comes from $10 webcam sensors covered in dust, mounted on vibrating factory machinery, streaming aggressively compressed MJPEG video over weak Wi-Fi.

If you do not simulate this violence during training, your $99\%$ accurate CNN will instantly drop to $30\%$ accuracy in production.

---

## 🎓 Socratic Deep Check
Ponder these questions before we begin:

> *"Mathematically, what destroys a CNN's feature maps when a clear PNG is converted to an aggressively compressed MJPEG video stream?"*

> *"CutMix pastes a region from Image B onto Image A with a label proportional to the pasted area. A student proposes using CutMix at TEST TIME: cut a region from a target image and paste it onto 10 different training images, then average the predictions of the augmented training images (a transductive augmentation). Why does this fail for object detection (where you need precise bounding boxes) but can theoretically work for image classification? What are the two practical failure modes that make test-time CutMix inferior to standard 10-crop TTA in practice?"*

---

## Table of Contents
1. **The MJPEG Assassin:** Discrete Cosine Transforms & Artifacts.
2. **FDA / FourierMix:** Style-Transfer in the Frequency Domain.
3. **Albumentations:** Building a brutal physical pipeline.
4. **AutoAugment and RandAugment:** Algorithmic simplification of policies.
5. **AugMix:** Mixing Augmentation Chains and JS-Consistency.
6. **Test-Time Augmentation (TTA):** The free accuracy boost.
7. **Class Imbalance Handling:** Explicit class distributions and Focal Loss.
8. **MixUp & CutMix Regularization:** Destroying the spatial prior.
9. **Mosaic Augmentation:** Scaling Object Detection.
10. **Adversarial Robustness:** FGSM, PGD, and 1-Pixel Vulnerabilities.


## 1. The MJPEG Assassin

Deep Neural Networks are essentially mathematical edge detectors. 
JPEG compression does not uniformly lower the resolution of an image. It slices the image into $8\times8$ blocks, applies a **Discrete Cosine Transform (DCT)**, and deletes the high-frequency color variations to save bytes.

When stitched back together, this creates microscopic "Block Artifacts"—sharp, artificial grid lines running across the entire image every 8 pixels. 

A human eye ignores these tiny grid lines. But a CNN's first layer is literally a sequence of edge detectors. The CNN latches onto these intense artificial vertical and horizontal lines, completely drowning out the actual physical edges of the objects you want to detect. The feature maps shatter.


In [ ]:
import albumentations as A
import cv2
from matplotlib import pyplot as plt
import numpy as np

# -----------------------------------------------------
# IMPLEMENTATION: The Brutal Edge Pipeline
# -----------------------------------------------------

try:
    # Albumentations v1 syntax
    noise_transform = A.GaussNoise(var_limit=(10.0, 50.0), p=0.3)
    # Test it compiles
    noise_transform(image=np.zeros((100, 100, 3), dtype=np.uint8))
except TypeError:
    # Albumentations v2+ syntax
    noise_transform = A.GaussNoise(std_limit=(10.0/255, 50.0/255), p=0.3)

production_transform = A.Compose([
    A.MotionBlur(blur_limit=15, p=0.3),
    noise_transform,
    A.RandomBrightnessContrast(brightness_limit=0.4, contrast_limit=0.4, p=0.5),
    A.ImageCompression(quality_lower=30, quality_upper=80, p=0.4),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=15, p=0.5)
])

# TEST IT
sample_img = np.zeros((256, 256, 3), dtype=np.uint8)
cv2.rectangle(sample_img, (50, 50), (200, 200), (0, 255, 0), -1)
augmented = production_transform(image=sample_img)['image']
print(f"Albumentations applied. Output shape: {augmented.shape}")


### COMMON PITFALLS
*   **Assuming resize interpolations are uniform:** Differing interpolation methods (`cv2.INTER_LINEAR` vs `cv2.INTER_NEAREST`) heavily alter edge sharpness. Using a mismatch between training interpolation and production resizing shatters accuracy.


## 2. FDA / FourierMix: Style Transfer in the Frequency Domain

While MJPEG artifacts corrupt high-frequency edges, **Fourier Domain Adaptation (FDA)** allows us to manipulate the low-frequency global statistics (the "style") without touching the high-frequency structural data (the "edges").

An image $x$ can be decomposed into its amplitude and phase using the Fast Fourier Transform (FFT):
$$F(x) = \mathcal{F}(x)$$
$$A(x) = |F(x)|, \quad P(x) = \angle F(x)$$

**The Fundamental Insight:** 
The **Phase** $P(x)$ carries the spatial/structural information (where the objects are). 
The **Amplitude** $A(x)$ carries the style/texture information (colors, lighting, global patterns).

**FourierMix:** By replacing a small window of the low-frequency amplitude spectrum of a source image with the amplitude from a target domain image, we effectively perform a brutal, mathematically perfect style transfer. The model learns to ignore global lighting shifts and focus entirely on the phase-preserved boundaries.

In [ ]:
import torch
import torch.fft

def fourier_mix(source_img, target_img, beta=0.1):
    """
    source_img, target_img: Tensors of shape (C, H, W)
    beta: Size of the low-frequency window to swap (0.0 to 1.0)
    """
    # 1. Compute 2D FFT
    fft_src = torch.fft.fft2(source_img)
    fft_trg = torch.fft.fft2(target_img)
    
    # 2. Shift low-frequencies to center for easy masking
    fft_src_shifted = torch.fft.fftshift(fft_src)
    fft_trg_shifted = torch.fft.fftshift(fft_trg)
    
    # 3. Extract Amplitude and Phase
    amp_src, pha_src = torch.abs(fft_src_shifted), torch.angle(fft_src_shifted)
    amp_trg = torch.abs(fft_trg_shifted)
    
    # 4. Define the low-frequency mask (center square)
    C, H, W = source_img.shape
    h_size, w_size = int(H * beta), int(W * beta)
    h_start, w_start = (H - h_size) // 2, (W - w_size) // 2
    
    # 5. Swap amplitude in the mask
    amp_src[:, h_start:h_start+h_size, w_start:w_start+w_size] = amp_trg[:, h_start:h_start+h_size, w_start:w_start+w_size]
    
    # 6. Reconstruct the image
    # A * exp(i * P)
    fft_src_mixed = amp_src * torch.exp(1j * pha_src)
    
    # Shift back and Inverse FFT
    fft_src_mixed = torch.fft.ifftshift(fft_src_mixed)
    mixed_img = torch.fft.ifft2(fft_src_mixed)
    
    return torch.real(mixed_img)

# TEST IT
src = torch.randn(3, 256, 256) # High-energy random noise
trg = torch.ones(3, 256, 256) * 0.5  # Flat grey style
mixed = fourier_mix(src, trg, beta=0.1)

print(f"FourierMix complete. Original dynamic range: {src.max()-src.min():.2f} | Mixed dynamic range: {mixed.max()-mixed.min():.2f}")
assert mixed.shape == src.shape, "Shape mismatch in FFT pipeline"


### COMMON PITFALLS
*   **Normalization Mismatch:** FFT must be applied on the raw pixel values (0-255 or 0-1). If you apply FDA *after* ImageNet normalization (which centers values around 0 with high variance), the spectral energy distribution collapses and the output image becomes garbage noise.

## 4. AutoAugment and RandAugment

Manually designing augmentation policies (as in the Albumentations pipeline) requires domain expertise and extensive hyperparameter tuning. **AutoAugment** uses reinforcement learning to search for the optimal augmentation policy—but this requires $\sim 15,000$ GPU hours to search on ImageNet.

**RandAugment** is the engineering simplification. It discards the reinforcement learning entirely. Instead, it randomly samples $N$ operations from a fixed set of 14 (AutoContrast, Equalize, Rotate, Solarize, Color, Posterize, Contrast, Brightness, Sharpness, ShearX, ShearY, TranslateX, TranslateY, Cutout) and applies them with a uniform magnitude $M$.

**TWO hyperparameters total:** 
- $N$ (how many ops)
- $M$ (how strong). 
*Default:* $N=2, M=9$. Zero GPU hours needed to search.

RandAugment consistently outperforms manually-designed pipelines by $\sim 0.5\%$ on ImageNet-C (corrupted ImageNet), which strictly measures robustness to distribution shift.


In [ ]:
import torch
import matplotlib.pyplot as plt

# -----------------------------------------------------
# IMPLEMENTATION: RandAugment
# -----------------------------------------------------
try:
    from torchvision.transforms import v2
    rand_augment = v2.RandAugment(num_ops=2, magnitude=9)
    print("Using torchvision v2 RandAugment")
except (ImportError, AttributeError):
    from torchvision.transforms import RandAugment
    rand_augment = RandAugment(num_ops=2, magnitude=9)
    print("Using torchvision v1 RandAugment")

# TEST IT
# Generate a dummy RGB image (CHW)
dummy_tensor = torch.randint(0, 256, (3, 224, 224), dtype=torch.uint8)

# Show diversity: generate 16 augmented versions
fig, axs = plt.subplots(4, 4, figsize=(8, 8))
for i in range(16):
    aug_img = rand_augment(dummy_tensor).permute(1, 2, 0).numpy()
    ax = axs[i//4, i%4]
    ax.imshow(aug_img)
    ax.axis('off')
plt.suptitle("RandAugment (N=2, M=9) Output Grid Diversity", y=1.02)
plt.tight_layout()
plt.show()

print("RandAugment generates highly diverse, randomized corruption samples instantly.")


### COMMON PITFALLS
*   **Applying magnitude $M$ blindly to everything:** Rotations and Translations scale with $M$ fine, but operations like `Solarize` easily destroy semantic information if $M$ is too high. Ensure the magnitude mapping functions are bounded safely.


## 5. AugMix: Mixing Augmentation Chains

Standard augmentations (like Albumentations or RandAugment) are applied **sequentially**—one operation corrupts the image, followed by another. The model only sees one highly corrupted image per training step.

**AugMix** applies $K=3$ SEPARATE chains of random augmentations to the same image. Then, it mixes them together with Dirichlet-sampled weights. The final generated image is a convex combination of the $K$ augmented versions PLUS the original clean image. This yields structurally preserved images that have highly diverse textures/colors.

**Jensen-Shannon Consistency Loss:**
During training with AugMix, the model computes predictions on the Clean image, and the Mixed images. The Jensen-Shannon Consistency Loss enforces that the neural network must output similar mathematical distributions for ALL variations. 
This explicitly penalizes the network heavily if its feature maps are sensitive to augmentation—a vastly stronger robustness signal than standard CE loss.

$$\text{Loss} = \text{CrossEntropy}(p_{clean}, y) + \lambda \cdot \text{JS}(p_{clean}, p_{mix1}, p_{mix2})$$


In [ ]:
import numpy as np

def apply_op(image, op_name, severity):
    # Dummy placeholder for augmented operations
    return image

def augmix_transform(image, severity=3, width=3, chain_depth=-1):
    """
    image: PIL Image or numpy uint8 array
    Returns the AugMix-transformed image.
    """
    ws = np.float32(np.random.dirichlet([1.0] * width))
    m = np.float32(np.random.beta(1.0, 1.0))
    
    mix = np.zeros_like(image, dtype=np.float32)
    for i in range(width):
        # Apply a chain of random augmentations
        image_aug = image.copy()
        depth = chain_depth if chain_depth > 0 else np.random.randint(1, 4)
        for _ in range(depth):
            op_name = np.random.choice([
                'autocontrast', 'equalize', 'rotate', 'solarize',
                'color', 'contrast', 'brightness', 'sharpness'
            ])
            image_aug = apply_op(image_aug, op_name, severity)
        mix += ws[i] * image_aug.astype(np.float32)
    
    # Convex combination of original and mixed
    mixed = (1 - m) * image.astype(np.float32) + m * mix
    return np.clip(mixed, 0, 255).astype(np.uint8)

# TEST IT
test_img = np.ones((64, 64, 3), dtype=np.uint8) * 128
aug_img = augmix_transform(test_img)
assert aug_img.shape == test_img.shape, "Shape mismatch"
assert aug_img.min() >= 0 and aug_img.max() <= 255, "Values out of bounds"
print("Augmix transform test passed!")

import torch.nn.functional as F

# -----------------------------------------------------
# IMPLEMENTATION: JS Consistency Loss for AugMix
# -----------------------------------------------------

def jensen_shannon_divergence(p, q, r):
    # p, q, r are logits. We convert to log_softmax and softmax probabilities
    p_probs = F.softmax(p, dim=1)
    q_probs = F.softmax(q, dim=1)
    r_probs = F.softmax(r, dim=1)
    
    # Calculate the mixture probability
    m = torch.clamp((p_probs + q_probs + r_probs) / 3.0, 1e-7, 1.0)
    
    # Compute KL divergences between each distribution and the mixture
    kl_p = F.kl_div(F.log_softmax(p, dim=1), m, reduction='batchmean')
    kl_q = F.kl_div(F.log_softmax(q, dim=1), m, reduction='batchmean')
    kl_r = F.kl_div(F.log_softmax(r, dim=1), m, reduction='batchmean')
    
    return (kl_p + kl_q + kl_r) / 3.0

# TEST IT
logits_clean = torch.randn(4, 10) # 4 samples, 10 classes
logits_aug1 = logits_clean + torch.randn(4, 10) * 0.5 
logits_aug2 = logits_clean + torch.randn(4, 10) * 0.8

js_loss = jensen_shannon_divergence(logits_clean, logits_aug1, logits_aug2)
# Standard CE loss would be added here
targets = torch.randint(0, 10, (4,))
ce_loss = F.cross_entropy(logits_clean, targets)

total_loss = ce_loss + 12.0 * js_loss # Lambda is typically high (e.g. 12)
print(f"CE: {ce_loss.item():.4f} | JS: {js_loss.item():.4f} | Total: {total_loss.item():.4f}")


### COMMON PITFALLS
*   **Heavy Compute Pipeline:** Generating multiple chains of augmentation dynamically heavily strains the dataloader's CPU workers. Without sufficient CPU multiprocessing, GPU utilization collapses waiting for Augmix batches.


## 6. Test-Time Augmentation (TTA)

Augmentations shouldn't just be for training. At inference, if the model predicts on exactly one central crop of an image, it is vulnerable to positional variation. 

**Test-Time Augmentation (TTA):** Generate $K$ conditionally augmented versions of the input image (e.g., center crop, 4 corners, horizontal flips), predict on all $K$ views, and mathematically average the Softmax probability vectors.

This provides a completely **free accuracy improvement** without any retraining. 
Benchmarking on ImageNet typically shows:
*   **5-crop TTA:** $\sim +0.5\%$ accuracy.
*   **10-crop TTA:** $\sim +1.2\%$ accuracy for ResNet-50.

**The Tradeoff:**
10-crop TTA requires 10 forward passes through the network. The inference latency increases by nearly $10\times$. You must balance the Accuracy/Latency Pareto curve strictly based on hardware constraints.


In [ ]:
# -----------------------------------------------------
# IMPLEMENTATION: TTA Inference Wrapper
# -----------------------------------------------------
import torchvision.transforms.functional as TF

def tta_predict_10_crop(model, image, crop_size=224):
    '''
    Applies standard 10-crop TTA (4 corners + 1 center) x (original + horizontal flip)
    '''
    model.eval()
    B, C, H, W = image.shape
    
    # Generate spatial boundaries for corner crops
    crops = [
        (0, 0),                                # Top left
        (0, W - crop_size),                    # Top right
        (H - crop_size, 0),                    # Bottom left
        (H - crop_size, W - crop_size),        # Bottom right
        ((H - crop_size)//2, (W - crop_size)//2) # Center
    ]
    
    predictions = []
    with torch.no_grad():
        for is_flipped in [False, True]:
            img_process = TF.hflip(image) if is_flipped else image
            for y, x in crops:
                crop = img_process[:, :, y:y+crop_size, x:x+crop_size]
                pred = torch.softmax(model(crop), dim=1)
                predictions.append(pred)
                
    # Average the 10 prediction vectors
    avg_prediction = torch.stack(predictions).mean(dim=0)
    return avg_prediction

# TEST IT
class DummyModel(torch.nn.Module):
    def forward(self, x): return torch.randn(x.size(0), 10) # 10 classes

dummy_img = torch.randn(1, 3, 256, 256)
model = DummyModel()
tta_out = tta_predict_10_crop(model, dummy_img, 224)

print(f"TTA Averaged Output Shape: {tta_out.shape} - Ready for argmax")


### COMMON PITFALLS
*   **Incompatible TTA Operators:** Applying Hue shifting or Scale jittering at TTA can completely break detection accuracy since you change the physical structure or color of the objects out of distribution. Typical TTA is strictly limited to deterministic spatial crops/flips or multi-scale pyramids.


## 7. Class Imbalance Handling

Real factory datasets have 10,000 "good product" images and 50 "defective" images. Standard training completely ignores the defectives. Here are three mathematical strategies to enforce minority visibility.

### 1. WeightedRandomSampler
Assign each sample a weight inversely proportional to class frequency. PyTorch's DataLoader will mathematically balance the sampling odds per epoch.

### 2. Focal Loss (Re-implemented)
Down-weights correctly classified background samples using an exponential decay $(1 - p_t)^\gamma$. For a 99% confident background prediction where $\gamma=2$, the loss weight becomes $(1 - 0.99)^2 = 0.0001$, clearing the gradient runway for the defective samples.

### 3. Class-Weighted Cross Entropy 
Using `nn.CrossEntropyLoss(weight=class_weights)`. Mathematically identical to oversampling minority classes, it multiplies the raw gradient of minority samples by $1.0 / \text{freq}$.


In [ ]:
from torch.utils.data import WeightedRandomSampler

# -----------------------------------------------------
# IMPLEMENTATION: Strategies
# -----------------------------------------------------

# 1. Sampler
class_counts = [1000, 10]
class_weights = [1.0 / count for count in class_counts]
sample_labels = [0]*1000 + [1]*10  # 1010 elements
sample_weights = [class_weights[label] for label in sample_labels]
sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

# 2. Focal Loss Context Module
class FocalLoss(torch.nn.Module):
    def __init__(self, alpha=1.0, gamma=2.0):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, inputs, targets):
        # inputs are logits
        ce_loss = torch.nn.functional.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss) 
        focal_loss = self.alpha * ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()

# 3. Class Weights Equivalent
tensor_class_weights = torch.tensor(class_weights, dtype=torch.float)
ce_weighted = torch.nn.CrossEntropyLoss(weight=tensor_class_weights)

# TEST IT 
logits = torch.randn(4, 2)
targets = torch.tensor([0, 0, 1, 0])
fl = FocalLoss()
print(f"Focal Loss Output: {fl(logits, targets).item():.4f}")
print(f"Weighted CE Loss: {ce_weighted(logits, targets).item():.4f}")

# Evaluation Note: Without handling, the confusion matrix for Class 1 (Minority) will be [0, 10] (100% False Negative Rate). With Focal Loss or Sampling, it shifts heavily toward correct diagonals.


### COMMON PITFALLS
*   **Balancing Validation Sets:** You ONLY balance the training dataloader. Balancing or resampling the validation set mathematically destroys your metrics since it no longer represents the real-world operational domain distributions.


## 8. MixUp & CutMix Regularization

**MixUp:** You take an image of a Dog, and an image of a Car. You mathematically blend them together (e.g., $60\%$ opacity Dog, $40\%$ opacity Car). The target label becomes exactly `[0.6 Dog, 0.4 Car]`. This forces the network to stop relying on binary "all or nothing" spatial features and learn continuous probability boundaries.

**CutMix:** You physically cut a square box out of the Car image and paste it directly on top of the Dog's face. The label is proportional to the area. 

**Why CutMix is genius:** If a CNN is trying to detect a Dog, it usually just learns to look for the Dog's eyes and ears. By pasting a car over the dog's face, the CNN is forced to look at the dog's paws, tail, and fur to make the prediction. It forces the network to learn holistic, full-body feature representations, acting as the ultimate spatial regularization.


In [ ]:
import numpy as np

# IMPLEMENTATION: CutMix Mathematical Core Calculation

def generate_cutmix_box(img_shape, lambda_val):
    B, C, H, W = img_shape
    cut_rat = np.sqrt(1. - lambda_val)
    cut_w = int(W * cut_rat)
    cut_h = int(H * cut_rat)

    # uniform
    cx = np.random.randint(W)
    cy = np.random.randint(H)

    bbx1 = np.clip(cx - cut_w // 2, 0, W)
    bby1 = np.clip(cy - cut_h // 2, 0, H)
    bbx2 = np.clip(cx + cut_w // 2, 0, W)
    bby2 = np.clip(cy + cut_h // 2, 0, H)

    return bbx1, bby1, bbx2, bby2

# TEST IT
B, C, H, W = 4, 3, 224, 224
lam = np.random.beta(1.0, 1.0) # Beta distribution scaling factor
bbx1, bby1, bbx2, bby2 = generate_cutmix_box((B, C, H, W), lam)
print(f"Generated CutMix paste coordinates: ({bbx1}, {bby1}) to ({bbx2}, {bby2}) for lambda {lam:.2f}")


### COMMON PITFALLS
*   **Cutting out the primary object entirely:** Since CutMix cuts randomly, you might entirely cut out a tiny insect and replace it with background from another class, but mathematically label it 60% insect still. This introduces false-label noise. The network tolerates it, but bounding box aware-cutting works better.


## 9. Mosaic Augmentation: Scaling Object Detection

**The Detection Bottleneck:**
Small objects are the bane of computer vision. In a standard ImageNet-sized crop ($224 \times 224$), a distant bird might occupy only $4 \times 4$ pixels. The CNN simply lacks enough sampling density to extract meaningful features.

**The Mosaic Solution (YOLOv4):**
Mosaic augmentation takes **4 different training images** and combines them into 1 grid at random center points. 

### Why does it solve Small Object Detection?
1.  **Increased Local Resolution:** By shrinking 4 images into the space of one, we effectively increase the context of the objects. It forces the model to detect objects at many different scales within a single forward pass.
2.  **Expanded Box Distribution:** Bounding boxes that were once at the edge of an image are now in the center of the "mosaic," providing the model with more diverse spatial priors.

### Batch Normalization & Statistics
Unlike **MixUp** (which blends pixel values) or **CutMix** (which pastes a block), Mosaic preserves the integrity of the pixels while drastically changing the **Batch-Normalization statistics**. 
Because 4 separate images are present in 1 sample, a mini-batch of 16 Mosaic images effectively provides the gradient stable signal of 64 different non-augmented images. This allows models to be trained with smaller batch sizes without the BN statistics becoming noisy and unstable.

---

## 🎓 Socratic Deep Check

> *"MixUp blends pixels to create 'ghost' distributions. Mosaic preserves pixels but shuffles spatial arrangements across 4 images. Why does Mosaic specifically alleviate the requirement for massive batch sizes typical of modern detectors, and why is it mathematically superior for Bounding Box regression compared to CutMix?"*


In [ ]:
# IMPLEMENTATION: Mosaic Grid Generator (Conceptual)
import numpy as np
import torch

def create_mosaic_sample(images, img_size=640):
    # images: list of 4 images [H, W, C]
    mosaic_img = np.full((img_size * 2, img_size * 2, 3), 114, dtype=np.uint8)
    
    # Random center for the 4-quadrant split
    yc, xc = [int(np.random.uniform(img_size // 2, 2 * img_size - img_size // 2)) for _ in range(2)]
    
    # Simple quadrant placement logic
    for i, img in enumerate(images):
        h, w, _ = img.shape
        # Quadrant 0: Top Left, 1: Top Right, etc.
        if i == 0: # top left
            mosaic_img[:yc, :xc] = img[:yc, :xc]
        elif i == 1: # top right
            mosaic_img[:yc, xc:] = img[:yc, :640-(2*640-xc)] # dummy crop
            
    return mosaic_img

# TEST IT
dummy_imgs = [np.random.randint(0, 255, (640, 640, 3), dtype=np.uint8) for _ in range(4)]
mosaic = create_mosaic_sample(dummy_imgs)
print(f"Mosaic Image Shape: {mosaic.shape}") # [1280, 1280, 3]
assert mosaic.shape == (1280, 1280, 3), "Mosaic scaling failed"


## 10. Adversarial Robustness: FGSM & PGD Training

While augmentations like MixUp or FDA handle natural distribution shifts, **Adversarial Augmentations** harden the model against intentional, malicious, or extreme edge-case perturbations.

**The 1-Pixel Vulnerability:** 
In a high-dimensional feature space, a CNN's decision boundary is often paper-thin. A specific, tiny perturbation $\epsilon$ can push an image of a "Cat" across the boundary into the "Toaster" class, even if the change is invisible to the human eye.

### 1. Fast Gradient Sign Method (FGSM)
FGSM calculates the gradient of the loss with respect to the input image, then moves the pixels in the direction of *maximum loss* by a step $\epsilon$.
$$\eta = \epsilon \cdot \text{sign}(\nabla_x J(\theta, x, y))$$
$$x_{adv} = x + \eta$$

### 2. Projected Gradient Descent (PGD)
PGD is essentially "Iterative FGSM" with a projection step. Instead of one large step, it takes $K$ small steps $\alpha$, and after each step, it projects the image back into an $\epsilon$-ball around the original image $x$.
$$x^{t+1} = \Pi_{x+S}(x^t + \alpha \cdot \text{sign}(\nabla_{x^t} J(\theta, x^t, y)))$$

**Adversarial Training:** By generating these perturbations *during training* and adding them to the mini-batch, we force the CNN to flatten its decision boundaries, making it robust to both noise and attacks.

In [ ]:
import torch.nn.functional as F

def generate_pgd_adversarial_batch(model, images, labels, eps=8/255, alpha=2/255, steps=4):
    """
    Generates an adversarial batch using PGD
    """
    adv_images = images.clone().detach()
    # Start at a random point in the epsilon ball for better exploration
    adv_images = adv_images + torch.empty_like(adv_images).uniform_(-eps, eps)
    adv_images = torch.clamp(adv_images, 0, 1)

    for _ in range(steps):
        adv_images.requires_grad = True
        outputs = model(adv_images)
        
        model.zero_grad()
        loss = F.cross_entropy(outputs, labels)
        loss.backward()
        
        # Move in sign-direction of gradient (maximize loss)
        grad = adv_images.grad.data
        adv_images = adv_images.detach() + alpha * grad.sign()
        
        # Projection step: clamp to epsilon ball and [0, 1] range
        delta = torch.clamp(adv_images - images, min=-eps, max=eps)
        adv_images = torch.clamp(images + delta, min=0, max=1).detach()
        
    return adv_images

# TEST IT
class MockModel(torch.nn.Module):
    def __init__(self): super().__init__(); self.conv = torch.nn.Conv2d(3, 10, 3)
    def forward(self, x): return self.conv(x).mean(dim=[2,3])

model = MockModel()
batch_imgs = torch.rand(4, 3, 32, 32)
batch_labels = torch.zeros(4).long()

adv_batch = generate_pgd_adversarial_batch(model, batch_imgs, batch_labels)
diff = torch.abs(adv_batch - batch_imgs).max().item()
print(f"PGD complete. Max perturbation: {diff:.4f} (Budget: {8/255:.4f})")
assert diff <= (8/255 + 1e-6), "Perturbation exceeded epsilon budget!"


### COMMON PITFALLS
*   **Label Leaking:** Using adversarial training with extremely high $\epsilon$ can modify the image so much that it actually *becomes* another class (e.g. a Cat becomes a Toaster), but you still label it a Cat. This introduces catastrophic noise into the training signal.
